# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Akash0-0/InternShip_Task01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** 1 — Ranking Signal Analysis · **Slice:** March 2026 page-month frame (my w03 contract) → April 2026 outcome
**Skills applied:** `building-baselines` (transparent rule, reason codes, precision@K) + `flyrank/flyrank-data` (warehouse layout, availability flags, panel warnings)

The Week-4 session showed where FlyRank's product flags come from — hand-written rules with honest thresholds (`needs_ctr_fix`, `is_quick_win`, refresh flags). This notebook rebuilds one such rule from observable signals on my lane's slice, checks that the signals it leans on are real, and reads the resulting top ten with a skeptic's eye. **This baseline is the number the Week-5 model must beat.**

## 1. My rule and its reason codes

**The rule in plain words:** "A page is worth a reviewer's time first if it still earns a lot of search impressions (volume), but its click-through rate is below what its position should earn (a CTR gap). The bigger the volume and the bigger the gap, the higher it ranks. Old pages (≥ 180 days) get a small tie-break bonus."

**Why this rule:** it is my rebuild of the `needs_ctr_fix` logic — a ranking-signal fix, which is exactly my lane. Position is knowable; CTR is knowable; the *gap* is the part a human can act on (title / meta / snippet).

**Reason codes (ONE per row):**

| code | meaning |
|---|---|
| `high_volume_ctr_gap` | score > 0: visible page whose CTR is below its position band's mean |
| `stale_high_volume_ctr_gap` | same, and the page is ≥ 180 days old |
| `no_actionable_gap` | score = 0: nothing to review |

**Action labels:** `review_listing` (score > 0) / `monitor` (score = 0).

Before trusting the rule, two signal checks — the assumptions it leans on must be real, or the rule is built on sand.

In [1]:
import os, json
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"          # keep the executed notebook clean
from pathlib import Path
import duckdb, pandas as pd, numpy as np

WORK_OUTPUTS = None
for _p in [Path.cwd(), *Path.cwd().parents]:        # anchor to the repo root (cwd varies by runner)
    if (_p / "work").is_dir() and (_p / ".git").exists():
        WORK_OUTPUTS = _p / "work" / "outputs"
        break
if WORK_OUTPUTS is None:
    WORK_OUTPUTS = Path.cwd() / "work" / "outputs"   # Colab edge case: falls back to cwd
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)
print("outputs dir:", WORK_OUTPUTS)

# Token: Colab Secret HF_TOKEN, else the huggingface_hub cache file. Never printed.
HF_TOKEN = os.environ.get("HF_TOKEN") or open(
    os.path.expanduser("~/.cache/huggingface/token")).read().strip()
assert HF_TOKEN, "Set HF_TOKEN (Colab Secret) or log in with huggingface-cli"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
# robust large-file reads: retry partial transfers instead of dying on flaky network
con.execute("SET http_retries=5")
con.execute("SET http_timeout=600")
con.execute("SET http_retry_wait_ms=2000")
con.execute("SET enable_http_metadata_cache=true")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# --- my slice (same contract as w03): page-month features from March, outcome from April ---
mar = con.sql(f"""
SELECT content_hash_id, client_hash_id,
       SUM(gsc_impressions)                                          AS imp_mar,
       SUM(gsc_clicks)                                               AS clk_mar,
       AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_pos_mar
FROM {FACT_MAR}
GROUP BY 1, 2
""").df()

apr = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr FROM {FACT_APR} GROUP BY 1").df()
dimc = con.sql(f"SELECT content_hash_id, content_created_date, content_type, word_count FROM {DIM_CONTENT}").df()

frame = mar.merge(apr, on="content_hash_id", how="left").merge(dimc, on="content_hash_id", how="left")
frame["imp_apr"] = frame["imp_apr"].fillna(0).astype(int)             # no April rows = zero observed impressions
frame["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(frame["content_created_date"])).dt.days
frame["ctr_mar"] = 100.0 * frame["clk_mar"] / frame["imp_mar"].replace(0, np.nan)   # % (rate columns are ×100)
frame["is_declining_next"] = ((frame["imp_mar"] > 0) & (frame["imp_apr"] < 0.8 * frame["imp_mar"])).astype(int)

# pages that actually had search visibility in March — the only rankable pages
vis = frame[frame["imp_mar"] > 0].copy()
vis["pos_band"] = pd.cut(vis["avg_pos_mar"], bins=[0, 3, 10, 20, 50, np.inf],
                         labels=["top_3", "page_1", "striking", "page_3_5", "deep"], right=True)
print(f"frame: {frame.shape} | visible pages (imp_mar > 0): {len(vis):,}")
print(f"base rate is_declining_next (visible): {vis['is_declining_next'].mean():.3f}")
frame[["content_hash_id", "client_hash_id", "imp_mar", "ctr_mar", "avg_pos_mar",
       "content_age_days", "is_declining_next"]].head()

outputs dir: C:\Users\palke\InternShip_Task01\work\outputs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: (331437, 12) | visible pages (imp_mar > 0): 176,738
base rate is_declining_next (visible): 0.532


,content_hash_id,client_hash_id,imp_mar,ctr_mar,avg_pos_mar,content_age_days,is_declining_next
0,content_4ed1708070b460a2,client_62f4a7e64f5e0096,10.0,0.000000,59.500000,195,0
1,content_8a911cc5f66adca0,client_62f4a7e64f5e0096,2.0,0.000000,63.500000,195,0
2,content_b3214c3bd821997d,client_62f4a7e64f5e0096,3802.0,0.473435,2.077452,195,0
3,content_d35ef59449784458,client_62f4a7e64f5e0096,55.0,0.000000,25.819674,195,1
4,content_13f098861dab35dd,client_62f4a7e64f5e0096,14.0,0.000000,41.611111,195,1


### Signal check 1 — CTR vs position (behind FlyRank's `needs_ctr_fix` logic)

The CTR-fix logic assumes **position drives CTR**: a page at the top of page one should earn a much higher CTR than a page buried on page three. If that is false, a "CTR gap" means nothing. Bucket visible pages by position band (the dictionary's `position_tier` bands) and compare mean CTR, with n per bucket.

In [2]:
sig1 = vis.groupby("pos_band", observed=True).agg(
    n=("ctr_mar", "count"),
    mean_ctr=("ctr_mar", "mean"),
    median_ctr=("ctr_mar", "median"),
    mean_imp=("imp_mar", "mean"),
).round(3)
print("pages with impressions but no position data (excluded from bands):",
      int(vis["pos_band"].isna().sum()))
sig1

pages with impressions but no position data (excluded from bands): 1434


,n,mean_ctr,median_ctr,mean_imp
pos_band,,,,
top_3,13136,1.101,0.096,2968.054
page_1,81619,0.515,0.000,1802.904
striking,32547,0.330,0.000,1073.431
page_3_5,34784,0.229,0.000,1650.433
deep,13218,0.098,0.000,164.156


**Verdict: CONFIRMED.** Mean CTR declines monotonically with worse position — 1.101% at top-3 → 0.098% beyond position 50, an ~11× drop. The CTR-fix assumption holds in this slice.

*Caveat, in the spirit of honest thresholds:* clicks are sparse — the **median** CTR is 0 for every band except top_3 (0.096). The mean is dragged up by a long tail of high-CTR pages. That is why the rule's expectation table uses the band **mean** (the norm a good page should reach) while acknowledging that many pages legitimately earn zero clicks in a month.

### Signal check 2 — staleness vs performance (behind the refresh flags)

The refresh flags assume **old content decays**: pages untouched for a long time rank worse and earn less. If that is false, age should not drive anything. Bucket pages by age tier (the dictionary's `age_tier`) and compare CTR, impressions, position, and the April-decline rate.

In [3]:
frame["age_tier"] = pd.cut(frame["content_age_days"], bins=[-1, 14, 30, 90, 180, 365, np.inf],
                                labels=["0-14", "15-30", "31-90", "91-180", "181-365", "365+"])
sig2 = frame.groupby("age_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr_mar", "mean"),
    mean_imp=("imp_mar", "mean"),
    mean_pos=("avg_pos_mar", "mean"),
    decl_rate=("is_declining_next", "mean"),
).round(3)
sig2

,n,mean_ctr,mean_imp,mean_pos,decl_rate
age_tier,,,,,
0-14,10965,0.425,94.129,17.469,0.040
15-30,15571,0.368,588.403,12.642,0.249
31-90,51718,0.385,1351.857,13.156,0.400
91-180,37735,0.319,1505.036,17.356,0.406
181-365,180833,0.621,608.118,19.160,0.233
365+,32491,0.301,1039.909,19.804,0.353


**Verdict: MIXED.** Position drifts worse with age from 15-30 days (mean 12.6) to 365+ (19.8) — except brand-new pages (0-14 days) that haven't ranked yet (17.5) — which mildly supports the refresh story. But CTR and decline rate are **non-monotonic**: the 181-365-day cohort has the *highest* mean CTR (0.621%) and the *lowest* decline rate (0.233) among cohorts with real traffic (mean ≥ 500 impressions). Older pages are not uniformly worse; some are evergreen winners.

**What this changes in my rule:** staleness does NOT drive the score. It only adds a 25% tie-break bonus (the `1 + 0.25 × stale` factor) — a stale page with a CTR gap is slightly more likely to need work, but signal 2 showed age alone is not a reliable flag. This is the "clearly-explained negative that just saved your rule" case: a refresh-only rule would have ranked the evergreen 181-365 cohort wrong.

## 2. Build the ranked queue (writes the CSV)

Encode the rule as a transparent score (hand-set, no fitted weights), attach one reason code and one action per row, rank everything, and write `work/outputs/baseline_action_score.csv`. Then evaluate honestly: precision@K against the April outcome (`is_declining_next`) with the **base rate** printed next to it. The label is used for **evaluation only** — never as an input.

In [4]:
# --- the rule: transparent, hand-set, no fitted weights ---
band_mean = vis.groupby("pos_band", observed=True)["ctr_mar"].mean()   # expectation table from signal check 1
vis["exp_ctr"] = vis["pos_band"].astype(str).map(band_mean).astype(float).fillna(0.0)
vis["gap"] = (vis["exp_ctr"] - vis["ctr_mar"]).clip(lower=0)           # how far below the band norm
vis["stale"] = (vis["content_age_days"] >= 180).astype(int)            # tie-break only (signal 2 = MIXED)
vis["visible"] = (vis["imp_mar"] >= 100).astype(int)                   # volume floor (quick-win precedent)

# score = visible * gap * (1 + 0.25*stale) * log1p(impressions)
vis["score"] = vis["visible"] * vis["gap"] * (1.0 + 0.25 * vis["stale"]) * np.log1p(vis["imp_mar"])

# ONE reason code + ONE action label per row
vis["reason_code"] = np.where(vis["score"] > 0,
                              np.where(vis["stale"] == 1, "stale_high_volume_ctr_gap", "high_volume_ctr_gap"),
                              "no_actionable_gap")
vis["action"] = np.where(vis["score"] > 0, "review_listing", "monitor")

queue = vis.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

# --- honest evaluation: precision@K vs base rate (label only for evaluation) ---
y = queue["is_declining_next"].values
prec = {k: float(y[:k].mean()) for k in (10, 50, 100)}
print("actionable (score > 0):", int((queue["score"] > 0).sum()))
print("reason codes:", queue["reason_code"].value_counts().to_dict())
print("actions:", queue["action"].value_counts().to_dict())
print(f"base rate (visible): {y.mean():.3f}")
for k, v in prec.items():
    print(f"precision@{k}: {v:.3f}")

# --- write the ranked queue (the CSV stays out of git by design; the notebook regenerates it) ---
csv_cols = ["rank", "content_hash_id", "client_hash_id", "score", "reason_code", "action",
            "imp_mar", "clk_mar", "ctr_mar", "avg_pos_mar", "pos_band", "exp_ctr", "gap",
            "content_age_days", "is_declining_next"]
queue[csv_cols].to_csv(WORK_OUTPUTS / "baseline_action_score.csv", index=False)
print("wrote", WORK_OUTPUTS / "baseline_action_score.csv")

# --- metrics receipt (this JSON IS committed) ---
metrics = {
    "rows": int(len(queue)),
    "actionable": int((queue["score"] > 0).sum()),
    "base_rate_visible": float(y.mean()),
    "precision_at_10": prec[10],
    "precision_at_50": prec[50],
    "precision_at_100": prec[100],
    "expectation_table_mean_ctr_by_band": {str(k): round(float(v), 4) for k, v in band_mean.items()},
    "top10_declining": int(y[:10].sum()),
    "client_concentration_top50": queue.head(50)["client_hash_id"].value_counts().head(5).to_dict(),
}
(WORK_OUTPUTS / "w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print("wrote", WORK_OUTPUTS / "w04_baseline_metrics.json")

queue.head()

actionable (score > 0): 81637
reason codes: {'no_actionable_gap': 95101, 'stale_high_volume_ctr_gap': 43454, 'high_volume_ctr_gap': 38183}
actions: {'monitor': 95101, 'review_listing': 81637}
base rate (visible): 0.532
precision@10: 0.800
precision@50: 0.840
precision@100: 0.820


wrote C:\Users\palke\InternShip_Task01\work\outputs\baseline_action_score.csv
wrote C:\Users\palke\InternShip_Task01\work\outputs\w04_baseline_metrics.json


,content_hash_id,client_hash_id,imp_mar,clk_mar,avg_pos_mar,imp_apr,content_created_date,content_type,word_count,content_age_days,...,is_declining_next,pos_band,exp_ctr,gap,stale,visible,score,reason_code,action,rank
0,content_306bc78dff1eb683,client_e547b89c05043229,80821.0,35.0,1.488604,29788,2025-03-21,keyword article,2528,375,...,1,top_3,1.100994,1.057689,1,1,14.939859,stale_high_volume_ctr_gap,review_listing,1
1,content_fc67675904376267,client_62f4a7e64f5e0096,60172.0,18.0,2.261303,19030,2025-05-21,keyword article,<NA>,314,...,1,top_3,1.100994,1.071080,1,1,14.734017,stale_high_volume_ctr_gap,review_listing,2
2,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,289.0,2.563756,191648,2025-03-21,keyword article,2895,375,...,0,top_3,1.100994,0.958977,1,1,14.652471,stale_high_volume_ctr_gap,review_listing,3
3,content_c46df0fa61530d86,client_e547b89c05043229,70398.0,42.0,1.556258,56908,2025-03-21,keyword article,2363,375,...,0,top_3,1.100994,1.041334,1,1,14.529121,stale_high_volume_ctr_gap,review_listing,4
4,content_9ef3d7516483e665,client_e547b89c05043229,89229.0,92.0,2.481596,59284,2025-03-21,keyword article,2624,375,...,1,top_3,1.100994,0.997889,1,1,14.218634,stale_high_volume_ctr_gap,review_listing,5


## 3. Top-10 review

Read the top ten with a skeptic's eye. For each row: the action, why it's there, and what would make it wrong. (The queue supports a top-20 review; the card asks for ten, hand-reviewed.)

In [5]:
top10 = queue.head(10)[["rank", "content_hash_id", "client_hash_id", "imp_mar", "clk_mar",
                        "ctr_mar", "avg_pos_mar", "pos_band", "exp_ctr", "gap",
                        "content_age_days", "score", "reason_code", "action", "is_declining_next"]]
top10.round(3)

,rank,content_hash_id,client_hash_id,imp_mar,clk_mar,ctr_mar,avg_pos_mar,pos_band,exp_ctr,gap,content_age_days,score,reason_code,action,is_declining_next
0,1,content_306bc78dff1eb683,client_e547b89c05043229,80821.0,35.0,0.043,1.489,top_3,1.101,1.058,375,14.940,stale_high_volume_ctr_gap,review_listing,1
1,2,content_fc67675904376267,client_62f4a7e64f5e0096,60172.0,18.0,0.030,2.261,top_3,1.101,1.071,314,14.734,stale_high_volume_ctr_gap,review_listing,1
2,3,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,289.0,0.142,2.564,top_3,1.101,0.959,375,14.652,stale_high_volume_ctr_gap,review_listing,0
3,4,content_c46df0fa61530d86,client_e547b89c05043229,70398.0,42.0,0.060,1.556,top_3,1.101,1.041,375,14.529,stale_high_volume_ctr_gap,review_listing,0
4,5,content_9ef3d7516483e665,client_e547b89c05043229,89229.0,92.0,0.103,2.482,top_3,1.101,0.998,375,14.219,stale_high_volume_ctr_gap,review_listing,1
5,6,content_b2b85c287474668d,client_e547b89c05043229,65304.0,61.0,0.093,1.542,top_3,1.101,1.008,351,13.964,stale_high_volume_ctr_gap,review_listing,1
6,7,content_66bf45eb0c5bb550,client_fef1a8f436438636,24259.0,1.0,0.004,2.785,top_3,1.101,1.097,393,13.843,stale_high_volume_ctr_gap,review_listing,1
7,8,content_7f52754cb72a5991,client_62f4a7e64f5e0096,43135.0,28.0,0.065,2.502,top_3,1.101,1.036,249,13.821,stale_high_volume_ctr_gap,review_listing,1
8,9,content_252aa5480bb1f8d7,client_73cda7b4e4f265ea,66698.0,75.0,0.112,2.394,top_3,1.101,0.989,371,13.726,stale_high_volume_ctr_gap,review_listing,1
9,10,content_1d7764b642f7bb9f,client_73cda7b4e4f265ea,23402.0,4.0,0.017,1.843,top_3,1.101,1.084,186,13.631,stale_high_volume_ctr_gap,review_listing,1


| # | action | why it's there | what would make it wrong |
|---|---|---|---|
| 1 | review_listing | 80,821 impressions at rank ~1.5 but CTR 0.043% vs the 1.101% top-3 norm — a ~26× gap, and the page already declined in April | the low CTR is a SERP-feature artifact (AI overview / video pack) that no title rewrite can fix |
| 2 | review_listing | 60,172 impressions at position 2.3 with 18 clicks (0.030%) — a top-3 listing earning almost nothing | the query's real CTR is structurally low (branded or zero-click queries), so the 1.1% benchmark doesn't apply |
| 3 | review_listing | biggest page in the queue (203,497 impressions); 0.142% CTR is ~8× below the norm → largest absolute opportunity | 0.14% may be its niche's norm, and it did NOT decline in April — weakest pick of the ten |
| 4 | review_listing | 70,398 impressions at position ~1.6 with 0.060% CTR — textbook listing problem IF intent matches | the keyword is a no-click query class (definitions, weather, scores) where CTR is low for everyone |
| 5 | review_listing | 89,229 impressions, top-3, CTR 0.103% (~10× below norm) AND already declining — both signals agree | the April decline is seasonal or algorithmic, not listing-related — fixing the title won't stop it |
| 6 | review_listing | 65,304 impressions at position 1.5 with 0.093% CTR; declining — same profile as #1 | impressions are inflated by broad keyword variants, so the band-norm denominator is wrong |
| 7 | review_listing | 24,259 impressions, position 2.8, ONE click all month (0.004%) — the most extreme gap; 393 days old | the page lost its SERP presence mid-March (legacy impressions, not live position), or one click is noise |
| 8 | review_listing | 43,135 impressions at top-3, CTR 0.065% (17× below norm), already declining | impressions are concentrated on a few volatile days, making the monthly CTR estimate unreliable |
| 9 | review_listing | 66,698 impressions, position 2.4, CTR 0.112%; 371 days old and declining | the client's GSC tracking under-counts clicks for this URL family (per-client instrumentation caveat) |
| 10 | review_listing | 23,402 impressions at position 1.8 with 4 clicks (0.017%); 186 days old, just past the staleness line | the SERP's shopping/featured block absorbs the clicks — verify before rewriting |

8 of 10 top rows were indeed declining in April (precision@10 = 0.80) — the top of the queue is not random, but every row still needs a human check before an action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# client concentration in the top of the queue — a fairness / data-quality lens
top50_clients = queue.head(50)["client_hash_id"].value_counts().head(6)
print("top-50 concentration by client:")
print(top50_clients.to_string())
print(f"\nthese {len(top50_clients)} clients hold {top50_clients.sum()} of 50 rows")

# weak picks: highest-ranked rows that did NOT decline in April (label = 0)
weak = queue[(queue["rank"] <= 10) & (queue["is_declining_next"] == 0)][
    ["rank", "content_hash_id", "imp_mar", "clk_mar", "ctr_mar", "avg_pos_mar", "score", "action"]]
print("\nweak picks (top-10 rows with no April decline):")
weak.round(3)

top-50 concentration by client:
client_hash_id
client_73cda7b4e4f265ea    25
client_e547b89c05043229    10
client_62f4a7e64f5e0096     7
client_fef1a8f436438636     6
client_23a62021009f63c4     2

these 5 clients hold 50 of 50 rows

weak picks (top-10 rows with no April decline):


,rank,content_hash_id,imp_mar,clk_mar,ctr_mar,avg_pos_mar,score,action
2,3,content_8d7d99f109e19aa2,203497.0,289.0,0.142,2.564,14.652,review_listing
3,4,content_c46df0fa61530d86,70398.0,42.0,0.060,1.556,14.529,review_listing


**Weak picks — where the rule could be wrong:**

1. **Rows 3 and 4** (`is_declining_next = 0`) are the clearest weak picks: the rule ranks them on a large CTR gap alone, with no April-decline evidence behind them. #3 is the biggest page in the queue (203k impressions) — if its 0.14% CTR is simply its niche's norm, the "fix" would buy nothing.
2. **Client concentration:** 25 of the top 50 belong to one client (`client_73cda7b4e4f265ea`); five clients hold all 50 rows. The queue may partly reflect those clients' tracking quality and volume mix, not universal page quality. A production queue should cap or diversify per client.
3. **All ten are `keyword article`s at top-3 positions.** The rule only scores pages that had March impressions, and the biggest gaps live at top-3 — but that also means the rule cannot see opportunity in pages that are *about to* enter page one.
4. **SERP-feature blindness:** a top-3 result can earn near-zero CTR because an AI overview, video, or shopping pack owns the clicks. The rule cannot see the SERP; the reviewer must check it before acting.

**Leakage check — no future-window or label-derived inputs:**
- Score inputs: `imp_mar`, `ctr_mar` (from March clicks/impressions), `avg_pos_mar`, `content_age_days` — all knowable by 2026-03-31. No April columns, no `trend_pct`/`trend_direction` (label sources), no `is_declining_next` as an input.
- `is_declining_next` appears only in (a) the precision@K evaluation and (b) the CSV as an annotation column for verification — it is never part of the score.
- The expectation table (`exp_ctr`) is the observed March mean CTR per position band — computed from the same feature window, no outcome involved.
- No product flags: this rule is my own rebuild of the `needs_ctr_fix` idea from observable signals — deliberately not a copy of FlyRank's shipped `health_score` / `priority_score` (those are not in the data anyway).

## 5. Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] Two signal verdicts with visible bucket tables and n — **CONFIRMED** (CTR vs position) and **MIXED** (staleness), both flag-linked
- [x] One rule with a score, a reason code, and an action label; ranked queue written to `work/outputs/baseline_action_score.csv`
- [x] Precision@K printed next to the base rate (p@10 = 0.80, p@50 = 0.84, p@100 = 0.82 vs 0.532 base rate)
- [x] Ten reviewed rows, each with action / why it's there / what would make it wrong
- [x] No future-window or label-derived inputs (leakage check in §4)
- [x] No client names, URLs, or private queries anywhere — only pseudonymized hashes
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.